# N3 — Escenarios metafóricos (Procesamiento)
## AI-MELT: Nivel 3 — Escenario metafórico

**Propósito:** Construir los escenarios metafóricos para cada metáfora convencional identificada en N2, con los cuatro componentes de MELT: secuencia narrativa, posicionamiento social, sesgo evaluativo y afecto.

**Entrada:**
- `n2_metaforas_convencionales.csv` (salida de N2)
- `n1_metaforas_primarias.csv` (para ejemplos textuales)
- `n1_correspondencias_ontologicas.csv`
- `n1_correspondencias_epistemicas.csv`

**Salida:**
- `n3_escenarios.csv` — un escenario por fila
- `n3_secuencia_narrativa.csv` — tres actos por escenario
- `n3_posicionamiento.csv` — grupos sociales y acciones
- `n3_sesgo_evaluativo.csv` — polos positivo/negativo con validación
- `n3_afecto.csv` — emociones facilitadas e inhibidas
- `n3_metadata.json`

**Visualización:** Ejecutar `N3_escenarios_viz.ipynb` después de este notebook.

---

### Estructura
1. Configuración
2. Carga de datos N1 y N2
3. Preparación de contexto por metáfora convencional
4. Generación de escenarios con LLM
5. Validación de sentimiento con pysentimiento
6. Clasificación de afecto con pysentimiento
7. Clasificación de estatus discursivo
8. Exportación

## 1. Configuración

In [ ]:
# ============================================================
# 1. DEPENDENCIAS E IMPORTS
# ============================================================
# Descomenta si necesitas instalar:
# !pip install anthropic openai pysentimiento spacy pandas tqdm

import gc
import json
import os
import re
import time
import warnings
from collections import Counter

import anthropic
import numpy as np
import openai
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

print("✓ Imports completados")

In [ ]:
# ============================================================
# 1B. CONFIGURACIÓN
# ============================================================

# ┌─────────────────────────────────────────────────────────┐
# │  MODIFICA ESTA SECCIÓN SEGÚN TUS NECESIDADES            │
# └─────────────────────────────────────────────────────────┘

# API Keys
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "sk-ant-...")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-...")

# Rutas
N1_DIR = "./outputs/N1/"
N2_DIR = "./outputs/N2/"
OUTPUT_DIR = "./outputs/N3/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Modelo LLM a usar para generación de escenarios
LLM_PROVIDER = "claude"   # "claude" o "openai"
CLAUDE_MODEL = "claude-sonnet-4-20250514"
OPENAI_MODEL = "gpt-4o"

# Parámetros
TEMPERATURE = 0.2          # Un poco más alta que N1 (tarea más creativa)
MAX_RETRIES = 3
RATE_LIMIT_PAUSE = 1.5
MAX_EXAMPLES_PER_MC = 10   # Máximo de ejemplos textuales por metáfora convencional

# Filtros
MIN_FRECUENCIA = 2         # Solo procesar metáforas convencionales con frecuencia >= N
ROBUSTEZ_FILTER = None     # None = todas; "ALTA" o "MODERADA" para filtrar

# Estatus discursivo: percentiles
ESTATUS_DOMINANTE_P = 90
ESTATUS_CHALLENGER_P = 70
ESTATUS_EMERGENTE_P = 40
# Por debajo del 40 → PERIFÉRICO

print("✓ Configuración cargada")
print(f"  LLM: {LLM_PROVIDER} ({CLAUDE_MODEL if LLM_PROVIDER == 'claude' else OPENAI_MODEL})")
print(f"  Filtro frecuencia mínima: {MIN_FRECUENCIA}")
print(f"  Filtro robustez: {ROBUSTEZ_FILTER or 'todas'}")

## 2. Carga de datos N1 y N2

In [ ]:
# ============================================================
# 2. CARGA DE DATOS
# ============================================================

# N2: Metáforas convencionales
df_conv = pd.read_csv(os.path.join(N2_DIR, "n2_metaforas_convencionales.csv"))
print(f"✓ n2_metaforas_convencionales.csv: {len(df_conv)} filas")

# N1: Metáforas primarias (para ejemplos textuales)
df_n1 = pd.read_csv(os.path.join(N1_DIR, "n1_metaforas_primarias.csv"))
print(f"✓ n1_metaforas_primarias.csv: {len(df_n1):,} filas")

# N1: Correspondencias
path_ont = os.path.join(N1_DIR, "n1_correspondencias_ontologicas.csv")
df_ont = pd.read_csv(path_ont) if os.path.exists(path_ont) else pd.DataFrame()
print(f"✓ correspondencias ontológicas: {len(df_ont):,}")

path_epi = os.path.join(N1_DIR, "n1_correspondencias_epistemicas.csv")
df_epi = pd.read_csv(path_epi) if os.path.exists(path_epi) else pd.DataFrame()
print(f"✓ correspondencias epistémicas: {len(df_epi):,}")

# N2: Tabla M:N para conectar convencionales con primarias
df_mn = pd.read_csv(os.path.join(N2_DIR, "n2_primaria_convencional.csv"))
print(f"✓ relaciones M:N: {len(df_mn):,}")

# Filtrar metáforas convencionales
df_work = df_conv.copy()
if MIN_FRECUENCIA > 1:
    df_work = df_work[df_work["frecuencia_absoluta"] >= MIN_FRECUENCIA]
if ROBUSTEZ_FILTER:
    df_work = df_work[df_work["robustez"] == ROBUSTEZ_FILTER]

print(f"\nMetáforas convencionales a procesar: {len(df_work)} (de {len(df_conv)} totales)")

## 3. Preparación de contexto por metáfora convencional

Para cada metáfora convencional se recopilan:
- Las correspondencias ontológicas y epistémicas de sus metáforas primarias
- Ejemplos textuales del corpus (expresión + contexto)

In [ ]:
# ============================================================
# 3. CONTEXTO POR METÁFORA CONVENCIONAL
# ============================================================

def gather_context_for_mc(mc_id, mc_name, df_mn, df_n1, df_ont, df_epi, max_examples):
    """Recopila correspondencias y ejemplos para una metáfora convencional."""

    # IDs de metáforas primarias asociadas
    prim_ids = df_mn[df_mn["ID_metaconvencional"] == mc_id]["ID_expresion"].tolist()

    # Ejemplos textuales
    df_prims = df_n1[df_n1["ID_expresion"].isin(prim_ids)]
    examples = []
    for _, row in df_prims.head(max_examples).iterrows():
        examples.append({
            "expresion": row.get("expresion_metaforica", ""),
            "contexto": row.get("contexto", ""),
            "foco": row.get("foco", ""),
            "dominio_fuente": row.get("dominio_fuente", ""),
            "dominio_meta": row.get("dominio_meta", ""),
        })

    # Correspondencias ontológicas
    corresp_ont = []
    if len(df_ont) > 0:
        df_ont_sub = df_ont[df_ont["ID_expresion"].isin(prim_ids)]
        for _, row in df_ont_sub.iterrows():
            corresp_ont.append({
                "elemento_fuente": row.get("elemento_fuente", ""),
                "elemento_meta": row.get("elemento_meta", ""),
                "evidencia": row.get("evidencia_textual", "")
            })

    # Correspondencias epistémicas
    corresp_epi = []
    if len(df_epi) > 0:
        df_epi_sub = df_epi[df_epi["ID_expresion"].isin(prim_ids)]
        for _, row in df_epi_sub.iterrows():
            corresp_epi.append({
                "tipo": row.get("tipo_inferencia", ""),
                "relacion_fuente": row.get("relacion_fuente", ""),
                "inferencia_meta": row.get("inferencia_meta", ""),
                "evidencia": row.get("evidencia_textual", "")
            })

    return {
        "n_primarias": len(prim_ids),
        "examples": examples,
        "corresp_ontologicas": corresp_ont,
        "corresp_epistemicas": corresp_epi,
    }


# Preparar contexto para todas las metáforas a procesar
contexts_by_mc = {}
for _, row in tqdm(df_work.iterrows(), total=len(df_work), desc="Preparando contexto"):
    mc_id = row["ID_metaconvencional"]
    mc_name = row["metafora_conceptual"]
    ctx = gather_context_for_mc(mc_id, mc_name, df_mn, df_n1, df_ont, df_epi, MAX_EXAMPLES_PER_MC)
    contexts_by_mc[mc_id] = ctx

print(f"\n✓ Contexto preparado para {len(contexts_by_mc)} metáforas convencionales")
total_examples = sum(len(c["examples"]) for c in contexts_by_mc.values())
total_ont = sum(len(c["corresp_ontologicas"]) for c in contexts_by_mc.values())
total_epi = sum(len(c["corresp_epistemicas"]) for c in contexts_by_mc.values())
print(f"  Ejemplos textuales: {total_examples}")
print(f"  Correspondencias ontológicas: {total_ont}")
print(f"  Correspondencias epistémicas: {total_epi}")

## 4. Generación de escenarios con LLM

Para cada metáfora convencional, el LLM recibe las correspondencias y ejemplos textuales y produce un JSON con los 4 componentes del escenario metafórico según MELT.

In [ ]:
# ============================================================
# 4A. PROMPT DE GENERACIÓN DE ESCENARIOS
# ============================================================

SYSTEM_PROMPT_N3 = """Eres un analista del discurso experto en la Teoría de la Metáfora Conceptual (Lakoff y Johnson, 1980) y en el modelo MELT (Metaphor Field-Loop Theory). Tu tarea es construir escenarios metafóricos a partir de metáforas convencionales identificadas en el Informe Final de la Comisión de la Verdad de Colombia.

Un ESCENARIO METAFÓRICO es una mini-narrativa que articula cuatro componentes:

1. SECUENCIA NARRATIVA: tres actos (inicio → desarrollo → desenlace) que describen la historia que la metáfora proyecta. Ejemplo: si la metáfora es LA SOCIEDAD ES UN CUERPO, la secuencia podría ser: inicio (el cuerpo sano), desarrollo (el cuerpo es herido), desenlace (el cuerpo sana o no).

2. POSICIONAMIENTO SOCIAL: qué grupos sociales participan en el escenario y qué acciones se les atribuyen. Indica si cada grupo es legitimado o deslegitimado por el escenario. Los grupos deben ser los actores reales del conflicto colombiano mencionados en el corpus (víctimas, perpetradores, Estado, comunidades, etc.).

3. SESGO EVALUATIVO: qué se presenta como positivo (polo positivo) y qué como negativo (polo negativo) en el escenario. Esta evaluación emerge de la metáfora, no de juicios externos.

4. AFECTO: qué emociones facilita y cuáles inhibe este escenario. Para cada emoción indica su función social (qué comportamiento colectivo promueve o desalienta). Incluye marcadores lingüísticos del corpus que evidencian ese afecto.

Responde SOLO con un JSON válido, sin texto adicional, sin markdown, sin backticks.
"""

USER_PROMPT_N3 = """Construye un escenario metafórico para la siguiente metáfora convencional del Informe Final de la Comisión de la Verdad de Colombia.

METÁFORA CONVENCIONAL: {metafora_conceptual}
DOMINIO FUENTE: {dominio_fuente}
DOMINIO META: {dominio_meta}
NÚMERO DE INSTANCIAS EN EL CORPUS: {frecuencia}

CORRESPONDENCIAS ONTOLÓGICAS:
{corresp_ont_text}

CORRESPONDENCIAS EPISTÉMICAS:
{corresp_epi_text}

EJEMPLOS TEXTUALES DEL CORPUS:
{examples_text}

Responde con un JSON con esta estructura exacta:
{{
  "nombre_escenario": "nombre descriptivo del escenario (frase breve)",
  "valoracion_uso": "POSITIVO|NEGATIVO|NEUTRO",
  "secuencia_narrativa": [
    {{"acto": "inicio", "descripcion": "situación inicial que la metáfora proyecta"}},
    {{"acto": "desarrollo", "descripcion": "acciones y procesos"}},
    {{"acto": "desenlace", "descripcion": "resultado o consecuencia"}}
  ],
  "posicionamiento": [
    {{
      "grupo_social": "nombre del grupo",
      "accion_atribuida": "qué hace o le pasa a este grupo en el escenario",
      "legitimacion": "legitimada|deslegitimada"
    }}
  ],
  "sesgo_evaluativo": {{
    "polo_positivo": "qué se presenta como bueno/deseable",
    "polo_negativo": "qué se presenta como malo/indeseable"
  }},
  "afectos": [
    {{
      "emocion": "nombre de la emoción",
      "tipo": "facilitado|inhibido",
      "funcion_social": "qué comportamiento colectivo promueve o desalienta",
      "marcadores_linguisticos": "expresiones del corpus que evidencian este afecto"
    }}
  ]
}}
"""

print("✓ Prompt de escenarios diseñado")

In [ ]:
# ============================================================
# 4B. FUNCIONES LLM
# ============================================================

if LLM_PROVIDER == "claude":
    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    MODEL = CLAUDE_MODEL
else:
    client = openai.OpenAI(api_key=OPENAI_API_KEY)
    MODEL = OPENAI_MODEL


def call_llm(user_prompt):
    """Llama al LLM configurado. Devuelve (resultado_json, tokens_in, tokens_out)."""
    for attempt in range(MAX_RETRIES):
        try:
            if LLM_PROVIDER == "claude":
                response = client.messages.create(
                    model=MODEL, max_tokens=4096, temperature=TEMPERATURE,
                    system=SYSTEM_PROMPT_N3,
                    messages=[{"role": "user", "content": user_prompt}])
                text = response.content[0].text.strip()
                tok_in, tok_out = response.usage.input_tokens, response.usage.output_tokens
            else:
                response = client.chat.completions.create(
                    model=MODEL, temperature=TEMPERATURE, max_tokens=4096,
                    response_format={"type": "json_object"},
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT_N3},
                        {"role": "user", "content": user_prompt}
                    ])
                text = response.choices[0].message.content.strip()
                tok_in, tok_out = response.usage.prompt_tokens, response.usage.completion_tokens

            text = re.sub(r'^```json\s*', '', text)
            text = re.sub(r'\s*```$', '', text)
            return json.loads(text), tok_in, tok_out

        except (json.JSONDecodeError, Exception) as e:
            if attempt < MAX_RETRIES - 1:
                time.sleep(3 * (attempt + 1))
            else:
                print(f"  ⚠ Error LLM: {e}")
    return None, 0, 0


def format_context_for_prompt(mc_row, context):
    """Formatea el contexto en texto para el prompt."""
    # Correspondencias ontológicas
    ont_lines = []
    for co in context["corresp_ontologicas"][:15]:
        ont_lines.append(f"  - {co['elemento_fuente']} ↔ {co['elemento_meta']}")
    ont_text = "\n".join(ont_lines) if ont_lines else "  (no disponibles)"

    # Correspondencias epistémicas
    epi_lines = []
    for ce in context["corresp_epistemicas"][:10]:
        epi_lines.append(f"  - [{ce['tipo']}] {ce['relacion_fuente']} → {ce['inferencia_meta']}")
    epi_text = "\n".join(epi_lines) if epi_lines else "  (no disponibles)"

    # Ejemplos textuales
    ex_lines = []
    for ex in context["examples"]:
        ex_lines.append(f"  • \"{ex['expresion']}\" (contexto: {ex['contexto'][:150]}...)")
    ex_text = "\n".join(ex_lines) if ex_lines else "  (no disponibles)"

    return USER_PROMPT_N3.format(
        metafora_conceptual=mc_row["metafora_conceptual"],
        dominio_fuente=mc_row["dominio_fuente"],
        dominio_meta=mc_row["dominio_meta"],
        frecuencia=mc_row["frecuencia_absoluta"],
        corresp_ont_text=ont_text,
        corresp_epi_text=epi_text,
        examples_text=ex_text,
    )


print(f"✓ LLM configurado: {LLM_PROVIDER} ({MODEL})")

### 4B-pre. Estimación de costo antes de procesar

Calcula cuántas llamadas API se harán y el costo estimado, para que puedas decidir si proceder.

In [ ]:
# ============================================================
# 4B-PRE. ESTIMACIÓN DE COSTO
# ============================================================

n_calls = len(df_work)
# Estimación: ~1500 tokens input + ~800 tokens output por llamada
est_tok_in = n_calls * 1500
est_tok_out = n_calls * 800

if LLM_PROVIDER == "claude":
    est_cost = (est_tok_in / 1e6 * 3) + (est_tok_out / 1e6 * 15)
else:
    est_cost = (est_tok_in / 1e6 * 2.5) + (est_tok_out / 1e6 * 10)

est_time = n_calls * (RATE_LIMIT_PAUSE + 2)  # ~2s por llamada + pausa

print("ESTIMACIÓN DE COSTO (ANTES DE EJECUTAR)")
print("=" * 50)
print(f"  Metáforas convencionales a procesar: {n_calls}")
print(f"  Tokens estimados: ~{est_tok_in:,} input + ~{est_tok_out:,} output")
print(f"  Costo estimado: ${est_cost:.2f} USD ({LLM_PROVIDER})")
print(f"  Tiempo estimado: ~{est_time/60:.1f} minutos")
print()
print("  ⚠ Si el costo es demasiado alto, ajusta MIN_FRECUENCIA o ROBUSTEZ_FILTER")
print("    para reducir el número de metáforas a procesar.")

In [ ]:
# ============================================================
# 4C. GENERACIÓN DE ESCENARIOS
# ============================================================

results_scenarios = []
total_tok_in = 0
total_tok_out = 0
t0 = time.time()

print(f"Generando escenarios para {len(df_work)} metáforas convencionales...")
print()

for _, mc_row in tqdm(df_work.iterrows(), total=len(df_work), desc="Escenarios"):
    mc_id = mc_row["ID_metaconvencional"]
    context = contexts_by_mc.get(mc_id, {"examples": [], "corresp_ontologicas": [], "corresp_epistemicas": []})

    prompt = format_context_for_prompt(mc_row, context)
    result, tok_in, tok_out = call_llm(prompt)
    total_tok_in += tok_in
    total_tok_out += tok_out

    if result:
        result["ID_metaconvencional"] = mc_id
        result["metafora_conceptual"] = mc_row["metafora_conceptual"]
        result["dominio_fuente"] = mc_row["dominio_fuente"]
        result["dominio_meta"] = mc_row["dominio_meta"]
        result["frecuencia"] = mc_row["frecuencia_absoluta"]
        results_scenarios.append(result)
    else:
        print(f"  ⚠ Sin resultado para: {mc_row['metafora_conceptual']}")

    time.sleep(RATE_LIMIT_PAUSE)

elapsed = time.time() - t0
costo = (total_tok_in / 1e6 * (3 if LLM_PROVIDER == "claude" else 2.5)) + \
        (total_tok_out / 1e6 * (15 if LLM_PROVIDER == "claude" else 10))

print(f"\n✓ Escenarios generados: {len(results_scenarios)} de {len(df_work)}")
print(f"  Tiempo: {elapsed:.1f}s")
print(f"  Tokens: {total_tok_in:,} in + {total_tok_out:,} out")
print(f"  Costo estimado: ${costo:.2f} USD")

## 5. Validación de sentimiento con pysentimiento

Se clasifican los contextos textuales de cada metáfora convencional con pysentimiento y se compara la polaridad detectada con el sesgo evaluativo generado por el LLM.

In [ ]:
# ============================================================
# 5. VALIDACIÓN DE SENTIMIENTO
# ============================================================

from pysentimiento import create_analyzer

print("Cargando modelo de sentimiento para español...")
sentiment_analyzer = create_analyzer(task="sentiment", lang="es")

# Para cada escenario, clasificar sentimiento de los contextos textuales
sentiment_validations = {}

for result in tqdm(results_scenarios, desc="Validando sentimiento"):
    mc_id = result["ID_metaconvencional"]
    context = contexts_by_mc.get(mc_id, {"examples": []})

    polarities = []
    for ex in context["examples"][:10]:
        ctx_text = ex.get("contexto", "")
        if ctx_text and len(ctx_text) > 10:
            try:
                pred = sentiment_analyzer.predict(ctx_text[:512])
                # pysentimiento devuelve POS, NEG, NEU con probabilities
                polarities.append({
                    "label": pred.output,
                    "pos": pred.probas.get("POS", 0),
                    "neg": pred.probas.get("NEG", 0),
                    "neu": pred.probas.get("NEU", 0),
                })
            except Exception:
                pass

    if polarities:
        avg_pos = np.mean([p["pos"] for p in polarities])
        avg_neg = np.mean([p["neg"] for p in polarities])
        avg_neu = np.mean([p["neu"] for p in polarities])
        dominant = max(["POS", "NEG", "NEU"], key=lambda x: {"POS": avg_pos, "NEG": avg_neg, "NEU": avg_neu}[x])
        sentiment_validations[mc_id] = {
            "avg_pos": round(float(avg_pos), 3),
            "avg_neg": round(float(avg_neg), 3),
            "avg_neu": round(float(avg_neu), 3),
            "dominant": dominant,
            "n_contexts": len(polarities)
        }

print(f"\n✓ Sentimiento validado para {len(sentiment_validations)} escenarios")

## 6. Clasificación de afecto con pysentimiento

Se usa el modelo de detección de emociones de pysentimiento sobre los contextos textuales para contrastar con las emociones generadas por el LLM.

In [ ]:
# ============================================================
# 6. CLASIFICACIÓN DE AFECTO (EMOCIONES)
# ============================================================

print("Cargando modelo de emociones para español...")
emotion_analyzer = create_analyzer(task="emotion", lang="es")

emotion_validations = {}

for result in tqdm(results_scenarios, desc="Detectando emociones"):
    mc_id = result["ID_metaconvencional"]
    context = contexts_by_mc.get(mc_id, {"examples": []})

    emotion_counts = Counter()
    for ex in context["examples"][:10]:
        ctx_text = ex.get("contexto", "")
        if ctx_text and len(ctx_text) > 10:
            try:
                pred = emotion_analyzer.predict(ctx_text[:512])
                emotion_counts[pred.output] += 1
            except Exception:
                pass

    if emotion_counts:
        emotion_validations[mc_id] = {
            "emociones_detectadas": dict(emotion_counts.most_common()),
            "emocion_dominante": emotion_counts.most_common(1)[0][0],
            "n_contexts": sum(emotion_counts.values())
        }

print(f"\n✓ Emociones detectadas para {len(emotion_validations)} escenarios")

# Liberar modelos
del sentiment_analyzer, emotion_analyzer
gc.collect()

## 7. Clasificación de estatus discursivo

Los escenarios se clasifican en cuatro categorías según su frecuencia y distribución textual:
- **Dominante:** alto impacto, muy distribuido
- **Challenger:** frecuente pero menos extendido
- **Emergente:** frecuencia moderada
- **Periférico:** baja frecuencia

In [ ]:
# ============================================================
# 7. ESTATUS DISCURSIVO
# ============================================================

# Calcular score compuesto para cada escenario
scores = []
for result in results_scenarios:
    mc_id = result["ID_metaconvencional"]
    mc_row = df_work[df_work["ID_metaconvencional"] == mc_id]
    if len(mc_row) == 0:
        scores.append(0)
        continue
    mc_row = mc_row.iloc[0]
    freq = mc_row.get("frecuencia_absoluta", 0)
    disp = mc_row.get("dispersion", 0)
    # Score compuesto: frecuencia normalizada * dispersión
    scores.append(freq * (1 + disp))

# Clasificar por percentiles
if scores:
    p_dom = np.percentile(scores, ESTATUS_DOMINANTE_P)
    p_cha = np.percentile(scores, ESTATUS_CHALLENGER_P)
    p_eme = np.percentile(scores, ESTATUS_EMERGENTE_P)

    for i, result in enumerate(results_scenarios):
        s = scores[i]
        if s >= p_dom:
            result["estatus_calculado"] = "DOMINANTE"
        elif s >= p_cha:
            result["estatus_calculado"] = "CHALLENGER"
        elif s >= p_eme:
            result["estatus_calculado"] = "EMERGENTE"
        else:
            result["estatus_calculado"] = "PERIFÉRICO"
        result["score_estatus"] = round(float(s), 2)

print("Distribución de estatus discursivo:")
estatus_counts = Counter(r.get("estatus_calculado", "?") for r in results_scenarios)
for est, count in sorted(estatus_counts.items()):
    print(f"  {est}: {count}")

## 8. Exportación

In [ ]:
# ============================================================
# 8A. CONSTRUIR Y EXPORTAR CSVs
# ============================================================

escenarios_rows = []
narrativa_rows = []
posicionamiento_rows = []
sesgo_rows = []
afecto_rows = []

for idx, result in enumerate(results_scenarios):
    esc_id = f"ESC-{idx+1:04d}"
    mc_id = result["ID_metaconvencional"]

    # Validaciones
    sent_val = sentiment_validations.get(mc_id, {})
    emo_val = emotion_validations.get(mc_id, {})

    # n3_escenarios.csv
    escenarios_rows.append({
        "ID_escenario": esc_id,
        "ID_metaconvencional_base": mc_id,
        "metafora_conceptual": result.get("metafora_conceptual", ""),
        "nombre_escenario": result.get("nombre_escenario", ""),
        "estatus": result.get("estatus_calculado", ""),
        "score_estatus": result.get("score_estatus", 0),
        "valoracion_uso": result.get("valoracion_uso", ""),
        "sentimiento_dominante_validacion": sent_val.get("dominant", ""),
        "emocion_dominante_validacion": emo_val.get("emocion_dominante", ""),
    })

    # n3_secuencia_narrativa.csv
    for acto in result.get("secuencia_narrativa", []):
        narrativa_rows.append({
            "ID_secuencia": f"SN-{esc_id}-{acto.get('acto', '')}",
            "ID_escenario": esc_id,
            "acto": acto.get("acto", ""),
            "descripcion": acto.get("descripcion", ""),
        })

    # n3_posicionamiento.csv
    for j, pos in enumerate(result.get("posicionamiento", [])):
        posicionamiento_rows.append({
            "ID_posicionamiento": f"PS-{esc_id}-{j+1:02d}",
            "ID_escenario": esc_id,
            "grupo_social": pos.get("grupo_social", ""),
            "accion_atribuida": pos.get("accion_atribuida", ""),
            "legitimacion": pos.get("legitimacion", ""),
        })

    # n3_sesgo_evaluativo.csv
    sesgo = result.get("sesgo_evaluativo", {})
    sesgo_rows.append({
        "ID_sesgo": f"SE-{esc_id}",
        "ID_escenario": esc_id,
        "polo_positivo": sesgo.get("polo_positivo", ""),
        "polo_negativo": sesgo.get("polo_negativo", ""),
        "sentimiento_pos_validacion": sent_val.get("avg_pos", None),
        "sentimiento_neg_validacion": sent_val.get("avg_neg", None),
        "sentimiento_neu_validacion": sent_val.get("avg_neu", None),
    })

    # n3_afecto.csv
    for k, af in enumerate(result.get("afectos", [])):
        afecto_rows.append({
            "ID_afecto": f"AF-{esc_id}-{k+1:02d}",
            "ID_escenario": esc_id,
            "emocion": af.get("emocion", ""),
            "tipo": af.get("tipo", ""),
            "funcion_social": af.get("funcion_social", ""),
            "marcadores_linguisticos": af.get("marcadores_linguisticos", ""),
        })

# Exportar
dfs_to_export = {
    "n3_escenarios.csv": pd.DataFrame(escenarios_rows),
    "n3_secuencia_narrativa.csv": pd.DataFrame(narrativa_rows),
    "n3_posicionamiento.csv": pd.DataFrame(posicionamiento_rows),
    "n3_sesgo_evaluativo.csv": pd.DataFrame(sesgo_rows),
    "n3_afecto.csv": pd.DataFrame(afecto_rows),
}

for fname, df in dfs_to_export.items():
    path = os.path.join(OUTPUT_DIR, fname)
    df.to_csv(path, index=False, encoding='utf-8-sig')
    print(f"✓ {fname} ({len(df):,} filas)")

## 8B-pre. Muestra para evaluación human-in-the-loop

Muestra aleatoria de escenarios para que el analista verifique:
- ¿La secuencia narrativa es coherente con la metáfora convencional?
- ¿El posicionamiento social identifica actores reales del conflicto?
- ¿Los afectos facilitados/inhibidos son pertinentes?

In [ ]:
# ============================================================
# 8B-PRE. MUESTRA HUMAN-IN-THE-LOOP
# ============================================================

EVAL_SAMPLE = min(30, len(results_scenarios))

if EVAL_SAMPLE > 0:
    # Tomar muestra del DataFrame de escenarios exportado
    df_esc_exported = pd.DataFrame(escenarios_rows)
    df_eval = df_esc_exported.sample(n=EVAL_SAMPLE, random_state=42).copy()

    # Añadir secuencia narrativa como texto
    df_narr_exported = pd.DataFrame(narrativa_rows)
    narr_by_esc = df_narr_exported.groupby("ID_escenario").apply(
        lambda x: " → ".join(x.sort_values("acto")["descripcion"].tolist())
    ).to_dict()
    df_eval["secuencia_narrativa_resumen"] = df_eval["ID_escenario"].map(narr_by_esc)

    # Columnas para evaluación manual
    df_eval["narrativa_coherente"] = ""          # Sí / No / Parcial
    df_eval["posicionamiento_correcto"] = ""     # Sí / No / Parcial
    df_eval["afectos_pertinentes"] = ""          # Sí / No / Parcial
    df_eval["observaciones"] = ""

    eval_path = os.path.join(OUTPUT_DIR, "n3_muestra_evaluacion.csv")
    df_eval.to_csv(eval_path, index=False, encoding='utf-8-sig')
    print(f"✓ {eval_path} ({EVAL_SAMPLE} escenarios)")
    print("  Columnas: narrativa_coherente, posicionamiento_correcto, afectos_pertinentes, observaciones")
else:
    print("⚠ Sin escenarios para muestra de evaluación.")

In [ ]:
# ============================================================
# 8B. METADATOS Y RESUMEN
# ============================================================

metadata = {
    "nivel": "N3",
    "llm_provider": LLM_PROVIDER,
    "model": MODEL,
    "temperature": TEMPERATURE,
    "metaforas_convencionales_entrada": len(df_work),
    "escenarios_generados": len(results_scenarios),
    "tokens_in": total_tok_in,
    "tokens_out": total_tok_out,
    "costo_usd": round(costo, 2),
    "tiempo_seg": round(elapsed, 1),
    "estatus_distribution": dict(estatus_counts),
}
with open(os.path.join(OUTPUT_DIR, "n3_metadata.json"), 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f"\n{'='*60}")
print("N3 — ESCENARIOS METAFÓRICOS COMPLETADO")
print(f"{'='*60}")
print(f"  Escenarios generados: {len(results_scenarios)}")
print(f"  Costo: ${costo:.2f} USD")
print(f"  Tiempo: {elapsed:.1f}s")
print(f"\n  Archivos en {OUTPUT_DIR}:")
for f_name in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f_name)) / 1024
    print(f"    📄 {f_name} ({size:.0f} KB)")
print("\n➡ SIGUIENTE: N3_escenarios_viz.ipynb para visualizaciones")
print("➡ LUEGO: N4_regimenes.ipynb")